In [1]:
import ee
import geemap
import pandas as pd
import geopandas as gpd
import numpy as np


In [2]:
# %% Initialize Earth Engine
ee.Initialize()

In [ ]:
# %% Load Study Area
nm_path = r"C:\Users\bsf31\Documents\data\NM\nm_vector.gpkg"
study_boundary = gpd.read_file(nm_path, layer='counties_dissolved')
silver_city_gdf = gpd.read_file(nm_path, layer='silver_city_edz')

In [4]:
# Convert to Earth Engine Geometry
ee_silver_city = geemap.geopandas_to_ee(silver_city_gdf).geometry()


In [6]:
# Define time range for NDVI calculation
start_date = '2024-05-01'
end_date = '2024-10-31'

# Load Sentinel-2 collection and filter
sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(ee_silver_city) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .median()  # Take median to reduce noise

# Compute NDVI: (NIR - Red) / (NIR + Red)
ndvi = sentinel2.expression(
    '(NIR - RED) / (NIR + RED)', {
        'NIR': sentinel2.select('B8'),  # Near Infrared Band
        'RED': sentinel2.select('B4')   # Red Band
    }
).rename('NDVI')

# Clip NDVI result to Silver City boundary
ndvi_silver_city = ndvi.clip(ee_silver_city)

# %% Visualization Parameters
ndvi_vis_params = {
    'min': -1,
    'max': 1,
    'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#ffffbf', '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850']
}

# %% Display on Map
Map = geemap.Map()
Map.centerObject(ee_silver_city, 12)
Map.addLayer(ndvi_silver_city, ndvi_vis_params, "NDVI Silver City")
Map

Map(center=[32.78946800998195, -108.27030955522709], controls=(WidgetControl(options=['position', 'transparent…

In [11]:
vis_params = {'bands': ['NDVI'], 'palette': ['#f7fcf5', '#f6fcf4', '#f6fcf4', '#f5fbf3', '#f5fbf2', '#f4fbf2', '#f4fbf1', '#f3faf0', '#f2faf0', '#f2faef', '#f1faee', '#f1faee', '#f0f9ed', '#f0f9ec', '#eff9ec', '#eff9eb', '#eef8ea', '#edf8ea', '#edf8e9', '#ecf8e8', '#ecf8e8', '#ebf7e7', '#ebf7e7', '#eaf7e6', '#e9f7e5', '#e9f7e5', '#e8f6e4', '#e8f6e3', '#e7f6e3', '#e7f6e2', '#e6f5e1', '#e5f5e1', '#e5f5e0', '#e4f5df', '#e3f4de', '#e2f4dd', '#e1f3dc', '#e0f3db', '#dff3da', '#def2d9', '#ddf2d8', '#dcf2d7', '#dbf1d6', '#dbf1d5', '#daf0d4', '#d9f0d3', '#d8f0d2', '#d7efd1', '#d6efd0', '#d5efcf', '#d4eece', '#d3eecd', '#d2edcc', '#d1edcb', '#d0edca', '#cfecc9', '#ceecc8', '#cdecc7', '#ccebc6', '#cbebc5', '#cbeac4', '#caeac3', '#c9eac2', '#c8e9c1', '#c7e9c0', '#c6e8bf', '#c4e8bd', '#c3e7bc', '#c2e7bb', '#c1e6ba', '#c0e6b9', '#bee5b8', '#bde5b6', '#bce4b5', '#bbe4b4', '#bae3b3', '#b8e3b2', '#b7e2b1', '#b6e2af', '#b5e1ae', '#b4e1ad', '#b2e0ac', '#b1e0ab', '#b0dfaa', '#afdfa8', '#aedea7', '#acdea6', '#abdda5', '#aadda4', '#a9dca3', '#a8dca2', '#a7dba0', '#a5db9f', '#a4da9e', '#a3da9d', '#a2d99c', '#a0d99b', '#9fd899', '#9ed798', '#9cd797', '#9bd696', '#99d595', '#98d594', '#97d492', '#95d391', '#94d390', '#92d28f', '#91d28e', '#90d18d', '#8ed08b', '#8dd08a', '#8bcf89', '#8ace88', '#88ce87', '#87cd86', '#86cc85', '#84cc83', '#83cb82', '#81ca81', '#80ca80', '#7fc97f', '#7dc87e', '#7cc87c', '#7ac77b', '#79c67a', '#78c679', '#76c578', '#75c477', '#73c476', '#72c375', '#70c274', '#6ec173', '#6dc072', '#6bc072', '#6abf71', '#68be70', '#66bd6f', '#65bd6f', '#63bc6e', '#62bb6d', '#60ba6c', '#5eb96b', '#5db96b', '#5bb86a', '#5ab769', '#58b668', '#56b567', '#55b567', '#53b466', '#52b365', '#50b264', '#4eb264', '#4db163', '#4bb062', '#4aaf61', '#48ae60', '#46ae60', '#45ad5f', '#43ac5e', '#42ab5d', '#40aa5d', '#3fa95c', '#3fa85b', '#3ea75a', '#3da65a', '#3ca559', '#3ba458', '#3aa357', '#39a257', '#38a156', '#37a055', '#369f54', '#359e53', '#349d53', '#339c52', '#329b51', '#319a50', '#309950', '#2f984f', '#2f974e', '#2e964d', '#2d954d', '#2c944c', '#2b934b', '#2a924a', '#29914a', '#289049', '#278f48', '#268e47', '#258d47', '#248c46', '#238b45', '#228a44', '#218944', '#208843', '#1f8742', '#1e8741', '#1d8640', '#1c8540', '#1a843f', '#19833e', '#18823d', '#17813d', '#16803c', '#157f3b', '#147e3a', '#137d39', '#127c39', '#117b38', '#107a37', '#0e7936', '#0d7836', '#0c7735', '#0b7734', '#0a7633', '#097532', '#087432', '#077331', '#067230', '#05712f', '#03702e', '#026f2e', '#016e2d', '#006d2c', '#006c2c', '#006b2b', '#00692a', '#00682a', '#006729', '#006529', '#006428', '#006328', '#006227', '#006027', '#005f26', '#005e26', '#005c25', '#005b25', '#005a24', '#005924', '#005723', '#005622', '#005522', '#005321', '#005221', '#005120', '#005020', '#004e1f', '#004d1f', '#004c1e', '#004a1e', '#00491d', '#00481d', '#00471c', '#00451c', '#00441b'], 'min': -1.0, 'max': 1.0}

# %% Display on Map
Map = geemap.Map()
Map.centerObject(ee_silver_city, 12)
Map.addLayer(ndvi_silver_city, vis_params, "NDVI Silver City")
Map

Map(center=[32.78946800998195, -108.27030955522709], controls=(WidgetControl(options=['position', 'transparent…

In [13]:
vis_params = {'bands': ['NDVI'], 'palette': ['#440154', '#440256', '#450457', '#450559', '#46075a', '#46085c', '#460a5d', '#460b5e', '#470d60', '#470e61', '#471063', '#471164', '#471365', '#481467', '#481668', '#481769', '#48186a', '#481a6c', '#481b6d', '#481c6e', '#481d6f', '#481f70', '#482071', '#482173', '#482374', '#482475', '#482576', '#482677', '#482878', '#482979', '#472a7a', '#472c7a', '#472d7b', '#472e7c', '#472f7d', '#46307e', '#46327e', '#46337f', '#463480', '#453581', '#453781', '#453882', '#443983', '#443a83', '#443b84', '#433d84', '#433e85', '#423f85', '#424086', '#424186', '#414287', '#414487', '#404588', '#404688', '#3f4788', '#3f4889', '#3e4989', '#3e4a89', '#3e4c8a', '#3d4d8a', '#3d4e8a', '#3c4f8a', '#3c508b', '#3b518b', '#3b528b', '#3a538b', '#3a548c', '#39558c', '#39568c', '#38588c', '#38598c', '#375a8c', '#375b8d', '#365c8d', '#365d8d', '#355e8d', '#355f8d', '#34608d', '#34618d', '#33628d', '#33638d', '#32648e', '#32658e', '#31668e', '#31678e', '#31688e', '#30698e', '#306a8e', '#2f6b8e', '#2f6c8e', '#2e6d8e', '#2e6e8e', '#2e6f8e', '#2d708e', '#2d718e', '#2c718e', '#2c728e', '#2c738e', '#2b748e', '#2b758e', '#2a768e', '#2a778e', '#2a788e', '#29798e', '#297a8e', '#297b8e', '#287c8e', '#287d8e', '#277e8e', '#277f8e', '#27808e', '#26818e', '#26828e', '#26828e', '#25838e', '#25848e', '#25858e', '#24868e', '#24878e', '#23888e', '#23898e', '#238a8d', '#228b8d', '#228c8d', '#228d8d', '#218e8d', '#218f8d', '#21908d', '#21918c', '#20928c', '#20928c', '#20938c', '#1f948c', '#1f958b', '#1f968b', '#1f978b', '#1f988b', '#1f998a', '#1f9a8a', '#1e9b8a', '#1e9c89', '#1e9d89', '#1f9e89', '#1f9f88', '#1fa088', '#1fa188', '#1fa187', '#1fa287', '#20a386', '#20a486', '#21a585', '#21a685', '#22a785', '#22a884', '#23a983', '#24aa83', '#25ab82', '#25ac82', '#26ad81', '#27ad81', '#28ae80', '#29af7f', '#2ab07f', '#2cb17e', '#2db27d', '#2eb37c', '#2fb47c', '#31b57b', '#32b67a', '#34b679', '#35b779', '#37b878', '#38b977', '#3aba76', '#3bbb75', '#3dbc74', '#3fbc73', '#40bd72', '#42be71', '#44bf70', '#46c06f', '#48c16e', '#4ac16d', '#4cc26c', '#4ec36b', '#50c46a', '#52c569', '#54c568', '#56c667', '#58c765', '#5ac864', '#5cc863', '#5ec962', '#60ca60', '#63cb5f', '#65cb5e', '#67cc5c', '#69cd5b', '#6ccd5a', '#6ece58', '#70cf57', '#73d056', '#75d054', '#77d153', '#7ad151', '#7cd250', '#7fd34e', '#81d34d', '#84d44b', '#86d549', '#89d548', '#8bd646', '#8ed645', '#90d743', '#93d741', '#95d840', '#98d83e', '#9bd93c', '#9dd93b', '#a0da39', '#a2da37', '#a5db36', '#a8db34', '#aadc32', '#addc30', '#b0dd2f', '#b2dd2d', '#b5de2b', '#b8de29', '#bade28', '#bddf26', '#c0df25', '#c2df23', '#c5e021', '#c8e020', '#cae11f', '#cde11d', '#d0e11c', '#d2e21b', '#d5e21a', '#d8e219', '#dae319', '#dde318', '#dfe318', '#e2e418', '#e5e419', '#e7e419', '#eae51a', '#ece51b', '#efe51c', '#f1e51d', '#f4e61e', '#f6e620', '#f8e621', '#fbe723', '#fde725'], 'min': -1.0, 'max': 1.0}
# %% Display on Map
Map = geemap.Map()
Map.centerObject(ee_silver_city, 12)
Map.addLayer(ndvi_silver_city, vis_params, "NDVI Silver City")
Map

Map(center=[32.78946800998195, -108.27030955522709], controls=(WidgetControl(options=['position', 'transparent…

In [14]:
# %% Export NDVI to Google Drive (Optional)
export_task_ndvi = ee.batch.Export.image.toDrive(
    image=ndvi_silver_city,
    description='NDVI_Silver_City_2023edz',
    folder='ndvi',
    fileNamePrefix='ndvi_silver_city_2023edz',
    region=ee_silver_city,
    scale=10,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e13
)
export_task_ndvi.start()